# 03 — Environmental feature engineering
Run one numbered step at a time. **No cells have been executed by the assistant.** No downloads, subprocesses or background jobs are used. All transformations are visible here.

Primary support: national 5 km grid, 2010–2023, end-of-year predictors for t+1 outcomes. Forest outputs describe surviving **year-2000 tree cover**, not full current natural forest. All scenario tables have threshold in their keys. A primary threshold is not approved automatically.

Sources: [Hansen GFC v1.12](https://storage.googleapis.com/earthenginepartners-hansen/GFC-2024-v1.12/download.html), [Hansen 2013](https://doi.org/10.1126/science.1244693), [fragmentation synthesis](https://doi.org/10.1126/sciadv.1500052), [Sri Lankan habitat study](https://doi.org/10.1155/2016/8148910). Sources justify measurements/mechanisms, not the exploratory thresholds below. Product revisions and the retrospective datamask mean these are hindcasts, not an archive of what was available in real time.

## Cell 1 — Imports and explicit settings
Requires pandas, numpy, geopandas, rasterio, scipy, pyproj. This cell only defines paths. Saving happens in Cell 13. Run the expensive fragmentation cell only when ready; progress is printed in the foreground.

In [1]:
from pathlib import Path
import json, hashlib
from contextlib import ExitStack
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from scipy import ndimage
from scipy.spatial.distance import cdist

ROOT = next(p for base in [Path.cwd(), *Path.cwd().parents]
            for p in [base, base / "components/habitat_risk"]
            if (p / "configs/study.json").exists())
CFG = json.loads((ROOT / "configs/study.json").read_text())
SP = ROOT / "data/interim/spatial"
OUT = ROOT / "data/processed/hri/wp5_v1"
GRID_ID = f"lka_laea_{CFG['grid_size_m']}m"
YEARS = list(range(2010, 2024))
THRESHOLDS = [10, 30, 50]  # exploratory sensitivity scenarios, NOT approved thresholds
PRIMARY_THRESHOLD = None  # set after inspecting sensitivity and reference evidence
THRESHOLD_RATIONALE = ""   # record source/reference comparison before selecting
MIN_COVERAGE = .95        # declared QA choice, sensitivity-test before release
BASELINE_YEARS = list(range(2010, 2015))  # early reference, not a climatological normal
PIXEL_M = 30
HALO_M = 1000             # local landscape context, NOT species dispersal distance
tables, metadata, audit = {}, {}, {}
input_paths = [ROOT / "configs/study.json"]


## Cell 2 — National scaffold and existing sources
Keep country cells regardless of bird observations. Polygon areas are projected land-intersection areas. Native-pixel area approximations can differ slightly at boundaries; record discrepancies rather than silently replacing denominators.

In [2]:
grid_path = SP / f"{GRID_ID}_grid.geojson"
grid = gpd.read_file(grid_path).to_crs(CFG["projected_crs"])
assert grid.cell_id.is_unique and grid.geometry.is_valid.all()
cells_index = pd.Index(grid.cell_id, name="cell_id")
land_area = pd.Series(grid.geometry.area.to_numpy(), index=cells_index)
scaffold = pd.MultiIndex.from_product([cells_index, YEARS],
    names=["cell_id", "reference_year"]).to_frame(index=False)
hist_path = SP / f"{GRID_ID}_canopy_loss_histogram.csv"
forest_path = SP / f"{GRID_ID}_forest_evidence.csv"
weather_path = SP / f"{GRID_ID}_weather_features.csv"
terrain_path = SP / f"{GRID_ID}_terrain_features.csv"
hist = pd.read_csv(hist_path)
fe = pd.read_csv(forest_path).set_index("cell_id").reindex(cells_index)
weather = pd.read_csv(weather_path)
terrain = pd.read_csv(terrain_path)
assert not hist.duplicated(["cell_id", "canopy_percent", "loss_year_code"]).any()
assert hist.loss_year_code.between(0, 24).all()
assert hist.canopy_percent.between(0, 100).all()
assert hist.mapped_land_area_m2.ge(0).all()
assert set(hist.cell_id) <= set(cells_index)
assert np.allclose(hist.groupby("cell_id").mapped_land_area_m2.sum().reindex(cells_index, fill_value=0),
                   fe.mapped_land_area_m2.fillna(0))
input_paths += [grid_path, hist_path, forest_path, weather_path, terrain_path]
print("National scaffold:", len(scaffold), "rows;", len(grid), "cells")


National scaffold: 40166 rows; 2869 cells


## Cell 3 — Forest history and threshold scenarios
Previous windows exclude t; annual loss includes t. Loss fractions use fixed eligible year-2000 forest area. No previous loss gives a missing age and a separate detection flag. Coverage combines source validity with sampled footprint/polygon agreement. Areas are mapped estimates, not statistically unbiased estimates.

In [3]:
def forest_history(h, threshold):
    base = h.loc[h.canopy_percent.ge(threshold)]
    baseline = base.groupby("cell_id").mapped_land_area_m2.sum().reindex(cells_index, fill_value=0)
    annual = (base.loc[base.loss_year_code.gt(0)]
              .groupby(["cell_id", "loss_year_code"]).mapped_land_area_m2.sum()
              .unstack(fill_value=0).reindex(index=cells_index, columns=range(1,25), fill_value=0))
    coverage = fe.observed_area_fraction_of_sampled_footprint
    footprint_ratio = fe.sampled_area_m2 / land_area
    ok = coverage.ge(MIN_COVERAGE) & footprint_ratio.between(.95, 1.05) & fe.mapped_land_area_m2.gt(0)
    out = []
    for year in YEARS:
        t = year - 2000
        prior = annual.loc[:, list(range(1, t))].sum(axis=1)
        current = annual[t]
        last = np.where(annual.loc[:, list(range(1,t))].to_numpy() > 0,
                        np.arange(2001, year)[None,:], np.nan)
        last = pd.DataFrame(last, index=cells_index).max(axis=1)
        d = pd.DataFrame({"baseline_canopy_percent": fe.mean_canopy_percent_on_mapped_land,
             "baseline_forest_ha": baseline/10000,
             "remaining_baseline_forest_proportion": (baseline-prior-current)/fe.mapped_land_area_m2,
             "annual_loss_ha": current/10000,
             "cumulative_prior_loss_ha": prior/10000,
             "years_since_previous_loss": year-last}, index=cells_index)
        for window in [1,3,5]:
            d[f"previous_{window}yr_loss_ha"] = annual.loc[:, list(range(t-window,t))].sum(axis=1)/10000
            d[f"previous_{window}yr_window_complete"] = year-window >= 2001
        for name in ["annual_loss_ha", "cumulative_prior_loss_ha", "previous_1yr_loss_ha",
                     "previous_3yr_loss_ha", "previous_5yr_loss_ha"]:
            d[name.replace("_ha","_fraction_of_baseline")] = d[name] / (baseline/10000).replace(0,np.nan)
        numeric = d.select_dtypes(include="number").columns
        d.loc[~ok, numeric] = np.nan
        d["previous_loss_detected"] = last.notna().where(ok).astype("boolean")
        d["forest_coverage"] = coverage
        d["sampled_to_polygon_area_ratio"] = footprint_ratio
        d["feature_status"] = np.where(ok, "coverage_pass", "insufficient_coverage")
        d["reference_year"] = year
        d["canopy_threshold_percent"] = threshold
        out.append(d.reset_index())
    return pd.concat(out, ignore_index=True)

forest = pd.concat([forest_history(hist, x) for x in THRESHOLDS], ignore_index=True)
tables["forest_threshold_scenarios"] = forest
metadata["forest_threshold_scenarios"] = {
    "source": "Hansen GFC 2024 v1.12; baseline 2000; loss 2001–2024",
    "key": ["cell_id","reference_year","canopy_threshold_percent"],
    "units": "ha, percent, fraction, years (column suffixes)",
    "denominator": "forest proportion: mapped valid land; loss fractions: threshold-specific baseline forest",
    "risk": "greater loss generally more pressure; condition requires species/land-cover context",
    "limitations": "no annual gains; tree cover includes plantations; native pixel centres; retrospective release"}
display(forest.head())


,cell_id,baseline_canopy_percent,baseline_forest_ha,remaining_baseline_forest_proportion,annual_loss_ha,cumulative_prior_loss_ha,years_since_previous_loss,previous_1yr_loss_ha,previous_1yr_window_complete,previous_3yr_loss_ha,...,cumulative_prior_loss_fraction_of_baseline,previous_1yr_loss_fraction_of_baseline,previous_3yr_loss_fraction_of_baseline,previous_5yr_loss_fraction_of_baseline,previous_loss_detected,forest_coverage,sampled_to_polygon_area_ratio,feature_status,reference_year,canopy_threshold_percent
0,lka_laea_5000m_-24_35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,...,NaN,NaN,NaN,NaN,<NA>,1.0,1.180258,insufficient_coverage,2010,10
1,lka_laea_5000m_-24_36,0.017578,0.000000,0.000000,0.0,0.0,NaN,0.0,True,0.0,...,NaN,NaN,NaN,NaN,False,1.0,1.047895,coverage_pass,2010,10
2,lka_laea_5000m_-23_4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,...,NaN,NaN,NaN,NaN,<NA>,1.0,1.612285,insufficient_coverage,2010,10
3,lka_laea_5000m_-23_6,0.073078,0.761669,0.002747,0.0,0.0,NaN,0.0,True,0.0,...,0.0,0.0,0.0,0.0,False,1.0,1.004908,coverage_pass,2010,10
4,lka_laea_5000m_-23_7,0.000000,0.000000,0.000000,0.0,0.0,NaN,0.0,True,0.0,...,NaN,NaN,NaN,NaN,False,1.0,1.046906,coverage_pass,2010,10


## Cell 4 — Threshold sensitivity and temporal leakage test
Compare all threshold scenarios rather than selecting the most convenient. Correlation is descriptive and is not evidence of ecological validity. Compare with independent reference forest mapping before setting PRIMARY_THRESHOLD. The test removes future loss events while preserving baseline area.

In [4]:
summary = forest.groupby(["canopy_threshold_percent","reference_year"]).agg(
    mapped_annual_loss_ha=("annual_loss_ha","sum"),
    cells_with_values=("annual_loss_ha","count"),
    mean_remaining_proportion=("remaining_baseline_forest_proportion","mean")).reset_index()
rank_input = forest.pivot(index=["cell_id","reference_year"],
                         columns="canopy_threshold_percent",
                         values="remaining_baseline_forest_proportion")
tables["forest_threshold_summary"] = summary
audit["threshold_rank_correlation"] = rank_input.corr(method="spearman").to_dict()
for t in [2010,2017,2023]:
    altered = hist.copy()
    altered.loc[altered.loss_year_code.gt(t-2000), "loss_year_code"] = 0
    # Duplicate histogram bins after alteration are valid for this aggregated test.
    original = forest_history(hist, 30).query("reference_year == @t").reset_index(drop=True)
    changed = forest_history(altered, 30).query("reference_year == @t").reset_index(drop=True)
    pd.testing.assert_frame_equal(original, changed)
if PRIMARY_THRESHOLD is not None:
    assert PRIMARY_THRESHOLD in THRESHOLDS and THRESHOLD_RATIONALE.strip()
    tables["forest_cell_year"] = forest.loc[forest.canopy_threshold_percent.eq(PRIMARY_THRESHOLD)].copy()
print("Future-loss perturbation tests passed. Primary threshold:", PRIMARY_THRESHOLD)
display(summary)


Future-loss perturbation tests passed. Primary threshold: None


,canopy_threshold_percent,reference_year,mapped_annual_loss_ha,cells_with_values,mean_remaining_proportion
0,10,2010,7343.582092,2850,0.608822
1,10,2011,15158.958300,2850,0.606622
2,10,2012,15104.485671,2850,0.604439
3,10,2013,8535.418876,2850,0.603188
4,10,2014,13230.676787,2850,0.601272
5,10,2015,7659.218885,2850,0.600143
6,10,2016,18554.250709,2850,0.597463
7,10,2017,13786.082208,2850,0.595457
8,10,2018,13977.560555,2850,0.593439
9,10,2019,10781.298495,2850,0.591889


## Cell 5 — Fragmentation helpers: local raster landscape
This implementation uses **cell-clipped patch areas** with 1 km surrounding context for labelling and isolation. Names explicitly describe this scale; they are not whole-landscape patch sizes. Four-neighbour connectivity is used. Edge length is counted only between valid forest and valid nonforest, not along crop borders or missing pixels. Mean nearest-patch distance is a **centroid proxy**, not edge-to-edge distance or functional connectivity.

The 30 m projected raster is an approximation of the native geographic pixels. Test raster resolution, halo size, canopy threshold and four/eight-neighbour choice before ecological interpretation. If only one contextual patch exists, neighbour distance is missing, not zero.

In [5]:
def read_context(geometry, sources):
    x0,y0,x1,y1 = geometry.buffer(HALO_M).bounds
    x0,y1 = np.floor(x0/PIXEL_M)*PIXEL_M, np.ceil(y1/PIXEL_M)*PIXEL_M
    width = int(np.ceil((x1-x0)/PIXEL_M)); height = int(np.ceil((y1-y0)/PIXEL_M))
    transform = from_origin(x0,y1,PIXEL_M,PIXEL_M)
    data = {band: np.full((height,width),255,dtype="uint8")
            for band in ["treecover2000","lossyear","datamask"]}
    for tile in sources:
        for band, src in tile.items():
            temp = np.full((height,width),255,dtype="uint8")
            reproject(rasterio.band(src,1),temp,src_transform=src.transform,src_crs=src.crs,
                      dst_transform=transform,dst_crs=grid.crs,dst_nodata=255,
                      resampling=Resampling.nearest)
            valid = temp != 255
            data[band][valid] = temp[valid]
    inside = geometry_mask([geometry],out_shape=(height,width),transform=transform,invert=True)
    return data, inside

def patch_metrics(data, inside, year, threshold):
    valid = (data["datamask"] == 1) & (data["treecover2000"] <= 100) & (data["lossyear"] <= 24)
    forest_mask = valid & (data["treecover2000"] >= threshold) & (
        (data["lossyear"] == 0) | (data["lossyear"] > year-2000))
    labels, count = ndimage.label(forest_mask, structure=ndimage.generate_binary_structure(2,1))
    ids, sizes = np.unique(labels[inside & forest_mask], return_counts=True)
    total = int((inside & valid).sum())
    coverage = total / inside.sum() if inside.any() else np.nan
    missing = {"cell_patch_count":np.nan,"mean_cell_patch_ha":np.nan,
               "largest_cell_patch_land_fraction":np.nan,"edge_m_per_land_ha":np.nan,
               "mean_nearest_patch_centroid_m":np.nan,"mapped_land_coverage":coverage}
    if total == 0 or coverage < MIN_COVERAGE: return missing
    edges = 0
    for axis in [0,1]:
        left = (slice(None,-1),slice(None)) if axis==0 else (slice(None),slice(None,-1))
        right = (slice(1,None),slice(None)) if axis==0 else (slice(None),slice(1,None))
        edges += ((forest_mask[left] != forest_mask[right]) &
                  valid[left] & valid[right] & inside[left] & inside[right]).sum()
    nearest = np.nan
    if len(ids) and count > 1:
        centres = np.array(ndimage.center_of_mass(forest_mask, labels, np.arange(1,count+1)))
        # Query only cell-intersecting patches against all contextual patches.
        distances = cdist(centres[ids-1], centres)*PIXEL_M
        distances[np.arange(len(ids)),ids-1] = np.inf
        nearest = float(distances.min(axis=1).mean())
    return {"cell_patch_count":len(ids),
            "mean_cell_patch_ha":sizes.mean()*PIXEL_M**2/10000 if len(ids) else np.nan,
            "largest_cell_patch_land_fraction":sizes.max()/total if len(ids) else 0.,
            "edge_m_per_land_ha":edges*PIXEL_M/(total*PIXEL_M**2/10000),
            "mean_nearest_patch_centroid_m":nearest,"mapped_land_coverage":coverage}


## Cell 6 — Run fragmentation in the foreground
This is the expensive step. Start with a small preview; preview outputs are never labelled national results. Set FRAGMENTATION_CELL_LIMIT=None yourself to process all cells. The raw tiles must already exist. No download occurs.

In [6]:
FRAGMENTATION_CELL_LIMIT = 5  # preview; set None for national processing
fragment_rows = []
with ExitStack() as stack:
    sources = []
    for tile in ["10N_070E","10N_080E"]:
        paths = {band: ROOT/f"data/raw/forest/Hansen_GFC-2024-v1.12_{band}_{tile}.tif"
                 for band in ["treecover2000","lossyear","datamask"]}
        assert all(p.exists() for p in paths.values()), paths
        input_paths += list(paths.values())
        sources.append({band:stack.enter_context(rasterio.open(p)) for band,p in paths.items()})
    chosen = grid if FRAGMENTATION_CELL_LIMIT is None else grid.head(FRAGMENTATION_CELL_LIMIT)
    for i, row in enumerate(chosen.itertuples(),1):
        data, inside = read_context(row.geometry,sources)
        for threshold in THRESHOLDS:
            for year in YEARS:
                fragment_rows.append({"cell_id":row.cell_id,"reference_year":year,
                    "canopy_threshold_percent":threshold,**patch_metrics(data,inside,year,threshold)})
        print(f"Fragmentation {i}/{len(chosen)} cells",flush=True)
frag = pd.DataFrame(fragment_rows)
frag_name = "fragmentation_threshold_scenarios" if FRAGMENTATION_CELL_LIMIT is None else "fragmentation_PREVIEW"
tables[frag_name] = frag
metadata[frag_name] = {"source":"Hansen v1.12 rasters","resolution_m":PIXEL_M,"halo_m":HALO_M,
    "key":["cell_id","reference_year","canopy_threshold_percent"],"connectivity":"4-neighbour",
    "risk":"edge exposure/isolation generally pressure; patch count is conditional on forest amount",
    "limitations":"cell-clipped areas; nearest centroid in bounded halo; raster resampling; no gains",
    "complete_national":FRAGMENTATION_CELL_LIMIT is None}
metric_cols = ["cell_patch_count","mean_cell_patch_ha","largest_cell_patch_land_fraction",
               "edge_m_per_land_ha","mean_nearest_patch_centroid_m"]
display(frag.groupby("canopy_threshold_percent")[metric_cols].corr(method="spearman"))
# Do not automatically drop correlated variables using held-out ML test data.


Fragmentation 1/5 cells
Fragmentation 2/5 cells
Fragmentation 3/5 cells
Fragmentation 4/5 cells
Fragmentation 5/5 cells


cell_patch_count  \
canopy_threshold_percent                                                      
10                       cell_patch_count                               1.0   
                         mean_cell_patch_ha                             NaN   
                         largest_cell_patch_land_fraction               1.0   
                         edge_m_per_land_ha                             1.0   
                         mean_nearest_patch_centroid_m                  NaN   
30                       cell_patch_count                               1.0   
                         mean_cell_patch_ha                             NaN   
                         largest_cell_patch_land_fraction               1.0   
                         edge_m_per_land_ha                             1.0   
                         mean_nearest_patch_centroid_m                  NaN   
50                       cell_patch_count                               NaN   
                         mean_cell_patch_ha                             NaN   
                         largest_cell_patch_land_fraction               NaN   
                         edge_m_per_land_ha                             NaN   
                         mean_nearest_patch_centroid_m                  NaN   

                                                           mean_cell_patch_ha  \
canopy_threshold_percent                                                        
10                       cell_patch_count                                 NaN   
                         mean_cell_patch_ha                               NaN   
                         largest_cell_patch_land_fraction                 NaN   
                         edge_m_per_land_ha                               NaN   
                         mean_nearest_patch_centroid_m                    NaN   
30                       cell_patch_count                                 NaN   
                         mean_cell_patch_ha                               NaN   
                         largest_cell_patch_land_fraction                 NaN   
                         edge_m_per_land_ha                               NaN   
                         mean_nearest_patch_centroid_m                    NaN   
50                       cell_patch_count                                 NaN   
                         mean_cell_patch_ha                               NaN   
                         largest_cell_patch_land_fraction                 NaN   
                         edge_m_per_land_ha                               NaN   
                         mean_nearest_patch_centroid_m                    NaN   

                                                           largest_cell_patch_land_fraction  \
canopy_threshold_percent                                                                      
10                       cell_patch_count                                               1.0   
                         mean_cell_patch_ha                                             NaN   
                         largest_cell_patch_land_fraction                               1.0   
                         edge_m_per_land_ha                                             1.0   
                         mean_nearest_patch_centroid_m                                  NaN   
30                       cell_patch_count                                               1.0   
                         mean_cell_patch_ha                                             NaN   
                         largest_cell_patch_land_fraction                               1.0   
                         edge_m_per_land_ha                                             1.0   
                         mean_nearest_patch_centroid_m                                  NaN   
50                       cell_patch_count                                               NaN   
                         mean_cell_patch_ha                                             NaN   
       

## Cell 7 — ERA5-Land annual climate and strictly trailing features
Reuse the checked annual source summaries. Upstream temperature is day-weighted Kelvin converted to °C. Precipitation uses the monthly-averaged reanalysis mean daily accumulation × days × 1000; do not repeat this conversion on annual millimetres.

The 2010–2014 reference is a short study baseline, not a climate normal. Anomalies are only produced for 2015 onward and only if all five baseline years are valid. Earlier years remain missing. Trailing five-year slopes require five complete years through t. Daily extremes are deferred.

In [7]:
assert not weather.duplicated(["cell_id","reference_year"]).any()
climate = scaffold.merge(weather,on=["cell_id","reference_year"],how="left",validate="one_to_one")
climate = climate.sort_values(["cell_id","reference_year"]).reset_index(drop=True)
def add_trailing_features(frame, value, prefix):
    frame = frame.copy()
    baseline = frame.loc[frame.reference_year.isin(BASELINE_YEARS)].groupby("cell_id")[value].agg(["mean","count"])
    ref = frame.cell_id.map(baseline["mean"].where(baseline["count"].eq(len(BASELINE_YEARS))))
    frame[prefix+"_anomaly"] = (frame[value]-ref).where(frame.reference_year.gt(max(BASELINE_YEARS)))
    frame[prefix+"_trailing5_slope_per_year"] = frame.groupby("cell_id")[value].transform(
        lambda s:s.rolling(5,min_periods=5).apply(lambda a:np.polyfit(np.arange(5),a,1)[0],raw=True))
    return frame
for value,prefix in [("annual_mean_temperature_c","temperature_c"),("annual_precipitation_mm","precipitation_mm")]:
    climate.loc[climate[value+"_coverage"].lt(MIN_COVERAGE),value] = np.nan
    climate = add_trailing_features(climate,value,prefix)
tables["climate_cell_year"] = climate
tables["terrain_cell"] = pd.DataFrame({"cell_id":cells_index}).merge(terrain,on="cell_id",how="left",validate="one_to_one")
metadata["climate_cell_year"] = {"source":"ERA5-Land existing audited annual summaries",
    "key":["cell_id","reference_year"],"units":"C, mm/year, anomaly in C or mm, slope per year",
    "baseline_years":BASELINE_YEARS,"anomalies_available_from":2015,
    "risk":"context only; no universal increasing/decreasing risk direction",
    "limitations":"0.1 degree source; partial coastal coverage; retrospective reanalysis; no daily extremes"}
metadata["terrain_cell"] = {"source":"existing terrain summary; see terrain_download_manifest.json",
    "key":["cell_id"],"units":"metres and degrees","risk":"static context; no generic risk direction"}
display(climate.head())


,cell_id,reference_year,annual_mean_temperature_c,annual_mean_temperature_c_coverage,annual_precipitation_mm,annual_precipitation_mm_coverage,weather_status,temperature_c_anomaly,temperature_c_trailing5_slope_per_year,precipitation_mm_anomaly,precipitation_mm_trailing5_slope_per_year
0,lka_laea_5000m_-10_-1,2010,26.099510,1.0,2045.684837,1.0,coverage_pass,NaN,NaN,NaN,NaN
1,lka_laea_5000m_-10_-1,2011,25.804277,1.0,1731.169643,1.0,coverage_pass,NaN,NaN,NaN,NaN
2,lka_laea_5000m_-10_-1,2012,26.343989,1.0,1185.748587,1.0,coverage_pass,NaN,NaN,NaN,NaN
3,lka_laea_5000m_-10_-1,2013,25.825384,1.0,1359.207929,1.0,coverage_pass,NaN,NaN,NaN,NaN
4,lka_laea_5000m_-10_-1,2014,26.067301,1.0,1879.975444,1.0,coverage_pass,NaN,-0.004331,NaN,-70.33805


## Cell 8 — Optional source acquisition register (no automatic downloads)
Supply approved **cell-level annual measurements** exported from an audited source workflow. The input is not just a filename: it needs its source/version, measurement period, availability cutoff, units, aggregation and QA description. Suitable starting catalogues: [MODIS MOD13Q1](https://lpdaac.usgs.gov/products/mod13q1v061/), [WorldPop](https://www.worldpop.org/), [GHSL](https://human-settlement.emergency.copernicus.eu/), [Protected Planet](https://www.protectedplanet.net/), [BirdLife](https://datazone.birdlife.org/). Verify licensing and actual year coverage before use.

For vegetation, quality-mask composites, apply provider scale factors, and aggregate pixels to land cells and composites to calendar years with coverage reporting. NDVI is bounded [-1,1]; clouds/missing values are not zero. For population counts, divide by land area; for population density, area-weight rather than sum densities. Built-up fraction uses mapped valid land. Do not interpolate sparse years silently. Current roads/pressure must be labelled static if historical snapshots are unavailable.

This optional interface validates source tables; it does **not** claim to implement provider-specific download and native-raster QA. Those acquisition workflows remain deferred until a product is chosen.

In [8]:
# Configure after a product and its upstream processing have been reviewed.
# Each spec: path, source_url, version, unit, value_column, aggregation, qa_method.
# Input columns: cell_id, reference_year, coverage_fraction, available_by_year, value_column.
OPTIONAL_ANNUAL = {}
# Example (not executed):
# OPTIONAL_ANNUAL["vegetation"] = dict(path=ROOT/"data/interim/spatial/annual_ndvi.csv",
#     source_url="...", version="...", unit="dimensionless", value_column="annual_mean_ndvi",
#     aggregation="quality-filtered temporal mean then valid-land area-weighted mean",
#     qa_method="provider QA masks and temporal coverage definition")
for name in ["vegetation","population","built_up","human_footprint"]:
    if name not in OPTIONAL_ANNUAL:
        audit[name] = "deferred: approved source and preprocessing not supplied"
        continue
    spec = OPTIONAL_ANNUAL[name]
    assert all(spec.get(k) for k in ["source_url","version","unit","value_column","aggregation","qa_method"])
    src = pd.read_csv(spec["path"])
    keys = ["cell_id","reference_year"]
    assert not src.duplicated(keys).any() and set(src.cell_id) <= set(cells_index)
    assert src.coverage_fraction.between(0,1).all()
    assert (src.available_by_year <= src.reference_year).all(), "Unavailable-at-cutoff values"
    joined = scaffold.merge(src,on=keys,how="left",validate="one_to_one").sort_values(keys)
    value = spec["value_column"]
    joined.loc[joined.coverage_fraction.lt(MIN_COVERAGE),value] = np.nan
    if name == "vegetation":
        assert joined[value].dropna().between(-1,1).all()
        joined = add_trailing_features(joined,value,"ndvi")
        joined["ndvi_negative_trend_exploratory"] = joined.ndvi_trailing5_slope_per_year.lt(0).where(
            joined.ndvi_trailing5_slope_per_year.notna()).astype("boolean")
    tables[name+"_cell_year"] = joined
    metadata[name+"_cell_year"] = {**{k:str(v) for k,v in spec.items()},"key":keys,
        "risk":"candidate only; NDVI decline needs seasonal/ecological validation; pressure is not observed bird response"}
    input_paths.append(Path(spec["path"]))


## Cell 9 — Optional conservation-area geometry
Provide a reviewed polygon layer for protected areas and/or IBAs. Dissolve overlapping polygons to avoid double counting. Output overlap uses cell land area; distance is minimum polygon-to-polygon distance (zero on intersection), not distance from every point in the cell. Without historical geometry, save static snapshot context only. Designation does not imply effective protection.

In [9]:
CONSERVATION = {}
# Example: CONSERVATION["protected_area"] = dict(path=ROOT/"data/raw/conservation/pa.gpkg",
#     source_url="...", version="...", snapshot_year=2026, scope="designated terrestrial sites")
for name in ["protected_area","iba"]:
    if name not in CONSERVATION:
        audit[name] = "deferred: licensed verified polygons unavailable"
        continue
    spec = CONSERVATION[name]
    assert all(spec.get(k) for k in ["source_url","version","snapshot_year","scope"])
    polygons = gpd.read_file(spec["path"])
    assert polygons.crs is not None and polygons.geometry.is_valid.all()
    assert polygons.geometry.geom_type.isin(["Polygon","MultiPolygon"]).all()
    union = polygons.to_crs(grid.crs).geometry.union_all()
    assert not union.is_empty
    out = pd.DataFrame({"cell_id":grid.cell_id,
        name+"_overlap_fraction":grid.geometry.intersection(union).area/grid.geometry.area,
        name+"_distance_m":grid.geometry.distance(union),
        "snapshot_year":spec["snapshot_year"]})
    tables[name+"_context_cell"] = out
    metadata[name+"_context_cell"] = {**{k:str(v) for k,v in spec.items()},"key":["cell_id"],
        "units":"fraction and metres","risk":"designation context, not automatic low risk",
        "limitations":"static snapshot; not a historical annual protection series"}
    input_paths.append(Path(spec["path"]))


## Cell 10 — Check every source before saving
Optional deferred sources have no zero-filled output. Tables are independent; scenario keys include threshold. National annual tables retain missing cells/years. Review preview results before requesting full fragmentation processing.

In [10]:
for name, frame in tables.items():
    if "cell_id" in frame:
        keys = ["cell_id"] + (["reference_year"] if "reference_year" in frame else [])
        if "canopy_threshold_percent" in frame: keys += ["canopy_threshold_percent"]
        assert not frame.duplicated(keys).any(), name
        assert set(frame.cell_id) <= set(cells_index), name
        if "reference_year" in frame:
            assert frame.reference_year.isin(YEARS).all(), name
        if name.endswith("_cell_year"):
            assert len(frame) == len(scaffold), (name,len(frame))
    for col in frame:
        if "fraction" in col or col.endswith("_proportion"):
            assert frame[col].dropna().between(-1e-8,1+1e-8).all(), (name,col)
    audit[name] = {"rows":len(frame),"missing_by_column":frame.isna().sum().to_dict()}
print("Table checks passed; inspect metadata and deferred items:")
display(pd.Series({k:str(v)[:180] for k,v in audit.items()}))


Table checks passed; inspect metadata and deferred items:


threshold_rank_correlation    {10: {10: 1.0, 30: 0.9973950300463368, 50: 0.9...
vegetation                    deferred: approved source and preprocessing no...
population                    deferred: approved source and preprocessing no...
built_up                      deferred: approved source and preprocessing no...
human_footprint               deferred: approved source and preprocessing no...
protected_area                 deferred: licensed verified polygons unavailable
iba                            deferred: licensed verified polygons unavailable
forest_threshold_scenarios    {'rows': 120498, 'missing_by_column': {'cell_i...
forest_threshold_summary      {'rows': 42, 'missing_by_column': {'canopy_thr...
fragmentation_PREVIEW         {'rows': 210, 'missing_by_column': {'cell_id':...
climate_cell_year             {'rows': 40166, 'missing_by_column': {'cell_id...
terrain_cell                  {'rows': 2869, 'missing_by_column': {'cell_id'...
dtype: object

## Cell 11 — Feature register and decision gates
The register includes units and interpretation from source metadata and coverage audits. Primary-threshold selection, national fragmentation execution and optional-source acquisition are separate gates. Scientific completion is not implied by successful file writing.

In [11]:
register_rows = []
for name, frame in tables.items():
    meta = metadata.get(name, metadata.get("forest_threshold_scenarios",{}))
    for column in frame.columns:
        if column in ["cell_id","reference_year","canopy_threshold_percent"]: continue
        register_rows.append({"table":name,"feature":column,"source":meta.get("source",meta.get("source_url","")),
            "units_and_definition":meta.get("units",meta.get("unit","see forest definitions")),
            "year_definition":"end of reference_year" if "reference_year" in frame else "static/source snapshot",
            "available_rows":int(frame[column].notna().sum()),"total_rows":len(frame),
            "risk_interpretation":meta.get("risk","diagnostic"),
            "limitations":meta.get("limitations","See source-specific cell and manifest")})
register = pd.DataFrame(register_rows)
completion = {
    "primary_canopy_threshold_selected":PRIMARY_THRESHOLD is not None and bool(THRESHOLD_RATIONALE.strip()),
    "national_fragmentation_processed":FRAGMENTATION_CELL_LIMIT is None,
    "vegetation": "available" if "vegetation_cell_year" in tables else "deferred",
    "human_pressure": "available" if any(n in OPTIONAL_ANNUAL for n in ["population","built_up","human_footprint"]) else "deferred",
    "conservation": "available" if CONSERVATION else "deferred",
    "daily_extremes":"deferred: annual tables insufficient",
    "climate_baseline":"2010–2014 reference; anomalies from 2015; not climate normal",
    "hri_scores":"not calculated"}
display(register)
display(pd.Series(completion))


,table,feature,source,units_and_definition,year_definition,available_rows,total_rows,risk_interpretation,limitations
0,forest_threshold_scenarios,baseline_canopy_percent,Hansen GFC 2024 v1.12; baseline 2000; loss 200...,"ha, percent, fraction, years (column suffixes)",end of reference_year,119700,120498,greater loss generally more pressure; conditio...,no annual gains; tree cover includes plantatio...
1,forest_threshold_scenarios,baseline_forest_ha,Hansen GFC 2024 v1.12; baseline 2000; loss 200...,"ha, percent, fraction, years (column suffixes)",end of reference_year,119700,120498,greater loss generally more pressure; conditio...,no annual gains; tree cover includes plantatio...
2,forest_threshold_scenarios,remaining_baseline_forest_proportion,Hansen GFC 2024 v1.12; baseline 2000; loss 200...,"ha, percent, fraction, years (column suffixes)",end of reference_year,119700,120498,greater loss generally more pressure; conditio...,no annual gains; tree cover includes plantatio...
3,forest_threshold_scenarios,annual_loss_ha,Hansen GFC 2024 v1.12; baseline 2000; loss 200...,"ha, percent, fraction, years (column suffixes)",end of reference_year,119700,120498,greater loss generally more pressure; conditio...,no annual gains; tree cover includes plantatio...
4,forest_threshold_scenarios,cumulative_prior_loss_ha,Hansen GFC 2024 v1.12; baseline 2000; loss 200...,"ha, percent, fraction, years (column suffixes)",end of reference_year,119700,120498,greater loss generally more pressure; conditio...,no annual gains; tree cover includes plantatio...
5,forest_threshold_scenarios,years_since_previous_loss,Hansen GFC 2024 v1.12; baseline 2000; loss 200...,"ha, percent, fraction, years (column suffixes)",end of reference_year,112924,120498,greater loss generally more pressure; conditio...,no annual gains; tree cover includes plantatio...
6,forest_threshold_scenarios,previous_1yr_loss_ha,Hansen GFC 2024 v1.12; baseline 2000; loss 200...,"ha, percent, fraction, years (column suffixes)",end of reference_year,119700,120498,greater loss generally more pressure; conditio...,no annual gains; tree cover includes plantatio...
7,forest_threshold_scenarios,previous_1yr_window_complete,Hansen GFC 2024 v1.12; baseline 2000; loss 200...,"ha, percent, fraction, years (column suffixes)",end of reference_year,120498,120498,greater loss generally more pressure; conditio...,no annual gains; tree cover includes plantatio...
8,forest_threshold_scenarios,previous_3yr_loss_ha,Hansen GFC 2024 v1.12; baseline 2000; loss 200...,"ha, percent, fraction, years (column suffixes)",end of reference_year,119700,120498,greater loss generally more pressure; conditio...,no annual gains; tree cover includes plantatio...
9,forest_threshold_scenarios,previous_3yr_window_complete,Hansen GFC 2024 v1.12; baseline 2000; loss 200...,"ha, percent, fraction, years (column suffixes)",end of reference_year,120498,120498,greater loss generally more pressure; conditio...,no annual gains; tree cover includes plantatio...


primary_canopy_threshold_selected                                                False
national_fragmentation_processed                                                 False
vegetation                                                                    deferred
human_pressure                                                                deferred
conservation                                                                  deferred
daily_extremes                                    deferred: annual tables insufficient
climate_baseline                     2010–2014 reference; anomalies from 2015; not ...
hri_scores                                                              not calculated
dtype: object

## Cell 12 — Final supervised review
Review threshold sensitivity, forest coverage, climate missingness, fragmentation scale and optional-source metadata. To make a national release, rerun Cell 6 with no cell limit, then Cells 10–13. Choose a threshold only after a documented reference comparison. Work Package 5 remains partial while these choices or required sources are unresolved.

The code below creates an optional combined *environmental* table only if a primary threshold is selected. It never joins species rows or uses bird-count coverage to select environmental cells.

In [12]:
if PRIMARY_THRESHOLD is not None:
    environment = scaffold.copy()
    selected = forest.loc[forest.canopy_threshold_percent.eq(PRIMARY_THRESHOLD)].drop(columns="canopy_threshold_percent")
    environment = environment.merge(selected,on=["cell_id","reference_year"],validate="one_to_one",how="left")
    climate_cols = [c for c in climate if c not in environment or c in ["cell_id","reference_year"]]
    environment = environment.merge(climate[climate_cols],on=["cell_id","reference_year"],validate="one_to_one",how="left")
    assert len(environment) == len(scaffold)
    tables["environmental_core_cell_year"] = environment
    metadata["environmental_core_cell_year"] = {"key":["cell_id","reference_year"],
        "source":"forest and climate tables only","limitations":"fragmentation and optional sources remain independent"}
else:
    print("Combined core table withheld until primary canopy threshold is justified.")


Combined core table withheld until primary canopy threshold is justified.


## Cell 13 — Save tables, metadata and audit (only when you execute)
Files go to a WP5-specific directory. Reruns replace these WP5 outputs. Large raster checksums are computed by streaming; this may take time in the foreground. All source inputs remain unchanged. No trained model or HRI label is produced.

In [13]:
OUT.mkdir(parents=True,exist_ok=True)
for name, frame in tables.items():
    frame.to_csv(OUT/f"{name}.csv",index=False)
register.to_csv(OUT/"environmental_feature_register.csv",index=False)
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b""):
            digest.update(block)
    return digest.hexdigest()
checksums = {}
for path in sorted(set(input_paths)):
    print("Hashing",path.name,flush=True)
    checksums[str(path.relative_to(ROOT))] = sha256_file(path)
report = {"grid_version":GRID_ID,"years":YEARS,"prediction_cutoff":"end of year t",
          "threshold_scenarios":THRESHOLDS,"primary_threshold":PRIMARY_THRESHOLD,
          "threshold_rationale":THRESHOLD_RATIONALE,"minimum_coverage":MIN_COVERAGE,
          "tables":metadata,"audit":audit,"completion":completion,"input_sha256":checksums}
(OUT/"metadata_and_audit.json").write_text(json.dumps(report,indent=2,default=str)+"\n")
print("Saved",OUT)
print("Review completion flags before calling WP5 complete:",completion)


Hashing study.json
Hashing lka_laea_5000m_canopy_loss_histogram.csv
Hashing lka_laea_5000m_forest_evidence.csv
Hashing lka_laea_5000m_grid.geojson
Hashing lka_laea_5000m_terrain_features.csv
Hashing lka_laea_5000m_weather_features.csv
Hashing Hansen_GFC-2024-v1.12_datamask_10N_070E.tif
Hashing Hansen_GFC-2024-v1.12_datamask_10N_080E.tif
Hashing Hansen_GFC-2024-v1.12_lossyear_10N_070E.tif
Hashing Hansen_GFC-2024-v1.12_lossyear_10N_080E.tif
Hashing Hansen_GFC-2024-v1.12_treecover2000_10N_070E.tif
Hashing Hansen_GFC-2024-v1.12_treecover2000_10N_080E.tif
Saved /Users/keerthikeerthi/Desktop/J26-IT-359/components/habitat_risk/data/processed/hri/wp5_v1
Review completion flags before calling WP5 complete: {'primary_canopy_threshold_selected': False, 'national_fragmentation_processed': False, 'vegetation': 'deferred', 'human_pressure': 'deferred', 'conservation': 'deferred', 'daily_extremes': 'deferred: annual tables insufficient', 'climate_baseline': '2010–2014 reference; anomalies from 2015; 